# 第五章 工具

# 1. Tools概述

## 1.1 工具的重要性

要构建更强大的AI工程应用，只有生成文本这样的“ 纸上谈兵 ”能力自然是不够的。

工具是赋予大语言模型 与外部世界交互能力 的关键组件，从而能让智能体执行搜索、计算、数据库查
询、邮件发送或调用第三方API等，进而构建功能强大的AI应用。借助工具，大模型才能从“ 认识世界 ”走向“ 改变世界 ”。

工具是构建智能体的核心要素之一！

## 1.2 工具调用的方式

在LangChain中，工具（Tools）实际上是指明确定义了输入和输出的 可调用函数 。因此， 工具调用(Tool Calling) 也被称为 函数调用(Function Calling) 。

具体有两种调用方式：

方式1：直接调用

这种方式，适合测试时使用。

In [2]:
from langchain_core.tools import tool
import langchain_core

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
        city: 城市名称，如"北京"、"上海"
        返回:天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"


result = get_weather.invoke("北京")
print(result)


北京晴天，温度 15°C


方式2：绑定到模型（主流）

这种方式，让AI来调用，开发中使用。

In [5]:

import asyncio
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from rich import print as rich_print
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致



model1= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

chat_model = ChatDeepSeek(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    model_name="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    参数:
        city: 城市名称，如"北京"、"上海"
        返回:天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"

# 绑定工具
model_with_tools = chat_model.bind_tools([get_weather])
# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print(response.content)
response = model_with_tools.invoke("2 + 3 = ？")
# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print(response.content)


AI 想调用工具： [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_tMVLNWqgclei72KGfgfJ9859', 'type': 'tool_call'}]
2 + 3 = 5


## 1.3 工具调用的整体流程

大模型能根据对话上下文决定何时调用工具以及传递哪些参数。

## 1.4 从Message流转看工具的调用

前提：模型的初始化


In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致



model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

这次改为不使用tool的用法

In [7]:
from langchain.messages import HumanMessage, ToolMessage
def get_weather(city: str):
    """
    获取天气的工具
    """
    return f"{city}天气晴朗"
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])
messages = [
    HumanMessage("今天北京天气如何")
]
# 模型生成调用工具请求
response = model_with_tools.invoke(messages)
# 添加AIMessage
messages.append(response)
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        tool_response = ToolMessage(content=get_weather(
                            **tool_call["args"]),
                            tool_call_id=tool_call["id"],
                            name="get_weather",
                        )
        messages.append(tool_response)


print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================

[{'arguments': '{"city":"北京"}', 'call_id': 'call_OhXtwtQ2GtJabeyexx9sZMd6', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_02412636ba9acd69016ab67cc5b89087d296fbba1de42a90d5', 'status': 'completed', 'internal_chat_message_metadata_passthrough': {'create_time': 1790344389.132053, 'turn_id': '01a0d8d7-55c2-7348-9008-cd2dbf04b94b'}, 'metadata': {'turn_id': '01a0d8d7-55c2-7348-9008-cd2dbf04b94b'}}]
Tool Calls:
  get_weather (call_OhXtwtQ2GtJabeyexx9sZMd6)
 Call ID: call_OhXtwtQ2GtJabeyexx9sZMd6
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗
=====================> messages <=====================
final_response: 
content=[{'type': 'text', 'text': '北京今天晴朗。', 'annotations': [], '

![](image/tool.png)

工具调用流程总结：

所以如果真正要大模型根据工具调用结果进行回复，完整的调用流程包括如下四个步骤：

步骤1：模型绑定工具 ：通过model.bind_tools([...])绑定一个或者多个工具。

步骤2：模型生成工具调用请求 ：用户输入问题，调用模型（比如invoke()）。如果需要调用工具，模
型返回包含工具调用信息（如工具名称和参数）的AIMessage。

步骤3：开发者手动执行工具 ：用户从响应中提取工具调用信息并手动调用对应的工具（比如工
具.invoke()）

步骤4：将工具执行结果ToolMessage传递给模型生成最终结果 ：将之前用户提问内容和手动执行工具
结果ToolMessage返回模型，模型最终生成回复。

> 特别注意：大模型调用工具是单次推理，直接响应，需要开发者手动执行工具并管理循环，适合简
单、确定的任务